# Exploratory Data Analysis
## Habit Drop Predictor — Final Year Project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

raw = pd.read_csv('../data/raw/90_day_habit_tracker.csv')
processed = pd.read_csv('../data/processed/real_user_processed.csv')
combined = pd.read_csv('../data/processed/combined_dataset.csv')

raw['Date'] = pd.to_datetime(raw['Date'])
processed['date'] = pd.to_datetime(processed['date'])

print('Raw dataset shape:', raw.shape)
print('Processed dataset shape:', processed.shape)
print('Combined dataset shape:', combined.shape)
raw.head()

## 1. Dataset Overview

In [ ]:
print('=== RAW DATASET INFO ===')
print(raw.dtypes)
print('\n=== MISSING VALUES ===')
print(raw.isnull().sum())
print('\n=== BASIC STATISTICS ===')
raw.describe()

## 2. Habit Completion Rates

In [ ]:
habits = ['Sleep', 'Workout', 'Reading', 'Screen_Limit', 'Budget', 'Journaling_Habit']
completion_rates = processed.groupby('habit')['completed'].mean() * 100
completion_rates = completion_rates.sort_values(ascending=False)

colors = ['#2ecc71' if v >= 70 else '#f39c12' if v >= 40 else '#e74c3c' 
          for v in completion_rates.values]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(completion_rates.index, completion_rates.values, color=colors, edgecolor='white')
ax.axhline(y=70, color='green', linestyle='--', alpha=0.7, label='Good threshold (70%)')
ax.axhline(y=40, color='orange', linestyle='--', alpha=0.7, label='Warning threshold (40%)')
for bar, val in zip(bars, completion_rates.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.set_title('Habit Completion Rates (90 Days)', fontsize=14, fontweight='bold')
ax.set_xlabel('Habit')
ax.set_ylabel('Completion Rate (%)')
ax.set_ylim(0, 115)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/completion_rates.png', dpi=150)
plt.show()
print('Saved: completion_rates.png')

## 3. Weekday vs Weekend Completion

In [ ]:
processed['day_type'] = processed['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})
day_comparison = processed.groupby(['habit', 'day_type'])['completed'].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(day_comparison.index))
width = 0.35
ax.bar(x - width/2, day_comparison['Weekday'], width, label='Weekday', color='#3498db', alpha=0.8)
ax.bar(x + width/2, day_comparison['Weekend'], width, label='Weekend', color='#e74c3c', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(day_comparison.index, rotation=15)
ax.set_title('Weekday vs Weekend Completion Rate per Habit', fontsize=14, fontweight='bold')
ax.set_ylabel('Completion Rate (%)')
ax.set_ylim(0, 120)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/weekday_vs_weekend.png', dpi=150)
plt.show()

## 4. Habit Completion Over Time

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()
habit_list = processed['habit'].unique()

for i, habit in enumerate(habit_list):
    hdf = processed[processed['habit'] == habit].sort_values('date')
    roll7 = hdf['completed'].rolling(7, min_periods=1).mean() * 100
    axes[i].bar(hdf['date'], hdf['completed']*100, alpha=0.3, color='steelblue', label='Daily')
    axes[i].plot(hdf['date'], roll7, color='navy', linewidth=2, label='7-day avg')
    axes[i].set_title(habit.replace('_', ' '), fontweight='bold')
    axes[i].set_ylim(0, 120)
    axes[i].set_ylabel('%')
    axes[i].legend(fontsize=8)
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Habit Completion Trends Over 90 Days', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/habit_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
corr_cols = ['Sleep_Hours', 'Workout_Duration_Min', 'Reading_Min',
             'Screen_Time_Hours', 'Daily_Expense (RM)', 'Mood_Score (1-10)']
corr_cols = [c for c in corr_cols if c in raw.columns]
corr_matrix = raw[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, mask=mask, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Between Lifestyle Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150)
plt.show()
print('\nKey correlations with Mood:')
if 'Mood_Score (1-10)' in corr_matrix.columns:
    mood_corr = corr_matrix['Mood_Score (1-10)'].drop('Mood_Score (1-10)').sort_values(ascending=False)
    print(mood_corr)

## 6. Mood vs Habit Completion

In [ ]:
if 'Mood_Score (1-10)' in raw.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(raw['Date'], raw['Mood_Score (1-10)'], color='purple',
            linewidth=2, label='Mood Score', marker='o', markersize=3)
    ax.axhline(y=raw['Mood_Score (1-10)'].mean(), color='purple',
               linestyle='--', alpha=0.5, label=f'Mean: {raw["Mood_Score (1-10)"].mean():.1f}')
    ax.fill_between(raw['Date'], raw['Mood_Score (1-10)'],
                    raw['Mood_Score (1-10)'].mean(), alpha=0.1, color='purple')
    ax.set_title('Mood Score Over 90 Days', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Mood Score (1-10)')
    ax.set_ylim(0, 11)
    ax.legend()
    plt.tight_layout()
    plt.savefig('../reports/figures/mood_trend.png', dpi=150)
    plt.show()

## 7. Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Class distribution
counts = combined['will_drop'].value_counts()
axes[0].pie(counts.values, labels=['Stable', 'Will Drop'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, explode=(0, 0.05))
axes[0].set_title('Target Variable Distribution', fontweight='bold')

# Drop events per habit
drop_by_habit = combined[combined['will_drop']==1]['habit'].value_counts()
axes[1].bar(drop_by_habit.index, drop_by_habit.values,
            color='#e74c3c', alpha=0.8, edgecolor='white')
axes[1].set_title('Drop Events by Habit Type', fontweight='bold')
axes[1].set_xlabel('Habit')
axes[1].set_ylabel('Number of Drop Events')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=150)
plt.show()
print(f'Total drop events: {counts[1]} ({counts[1]/len(combined)*100:.1f}%)')
print(f'Total stable:      {counts[0]} ({counts[0]/len(combined)*100:.1f}%)')

## 8. Feature Distributions

In [ ]:
feature_cols = ['roll7_rate', 'streak', 'days_since_miss', 'habit_age']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    stable   = combined[combined['will_drop']==0][col]
    dropping = combined[combined['will_drop']==1][col]
    axes[i].hist(stable,   bins=30, alpha=0.6, color='green', label='Stable',    density=True)
    axes[i].hist(dropping, bins=30, alpha=0.6, color='red',   label='Will Drop', density=True)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    axes[i].legend()

plt.suptitle('Feature Distributions: Stable vs Will Drop', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=150)
plt.show()

## 9. Key Findings Summary

In [ ]:
print('='*60)
print('  KEY FINDINGS FROM EDA')
print('='*60)
print(f'\n1. Dataset: 90 days of habit tracking across 6 habits')
print(f'2. Total records (with synthetic): {len(combined):,}')
print(f'3. Drop event rate: {combined["will_drop"].mean()*100:.1f}% (class imbalance present)')
cr = processed.groupby('habit')['completed'].mean()*100
print(f'4. Highest completion: {cr.idxmax()} ({cr.max():.1f}%)')
print(f'5. Lowest completion:  {cr.idxmin()} ({cr.min():.1f}%)')
if 'Mood_Score (1-10)' in raw.columns:
    print(f'6. Average mood score: {raw["Mood_Score (1-10)"].mean():.1f}/10')
wd = processed.groupby('is_weekend')['completed'].mean()*100
print(f'7. Weekday completion: {wd[0]:.1f}% | Weekend: {wd[1]:.1f}%')
print('\nAll EDA figures saved to reports/figures/')
print('='*60)